# FrameWeave v6 multi-seed — Qwen2.5-VL-7B HERBench eval

Run All scores the three checkpoints from the train notebook on HERBench lite_v2. Seeds **1, 100, 2027**. Methods: `global_timefix`, `hybrid_noglobal`, `hybrid`.

Predictions resume from Drive. The last cell writes per-seed scores and mean ± sample standard deviation.

Run All clones [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) into `/content/FrameWeave` and links `reljoin/` plus the runners into `/content`. Do not upload `.py` files. This notebook does not train.


## 1. Configuration


In [ ]:
# 1. Eval-only config — seeds 1, 100, 2027
from __future__ import annotations
from pathlib import Path

SEEDS = [1, 100, 2027]
METHODS = "global_timefix,hybrid_noglobal,hybrid"
CKPT_NAMES = ["GlobalTimeFix.pt", "HybridNoGlobal.pt", "Hybrid.pt"]

ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."
OUT_ROOT = ROOT / "outputs_reljoin_v6_qwen_multiseed"
PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
print({"SEEDS": SEEDS, "METHODS": METHODS, "OUT_ROOT": str(OUT_ROOT), "PERSIST": str(PERSIST)})


## 2. Environment (`qwen_env`)


In [ ]:
# 2. Bootstrap isolated qwen_env (Python 3.11 + torch 2.4.0 + transformers 4.51.3)
from __future__ import annotations
import os, subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "qwen_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs_reljoin_v6_qwen_multiseed" / "logs" / "cell4_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc

sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "pyarrow==25.0.1", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging", "qwen-vl-utils",
])
print("Installing qwen-vl-utils; flash-attn is optional (SDPA is enough)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
    check=False,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, qwen_vl_utils; "
    "print(torch.__version__); print(transformers.__version__); print(qwen_vl_utils.__file__); "
    "print(torch.__file__); print(transformers.__file__); "
    "import importlib.util, numpy, pyarrow; print('numpy', numpy.__version__); print('pyarrow', pyarrow.__version__); assert numpy.__version__.startswith('1.'), numpy.__version__; assert pyarrow.__version__ == '25.0.1', pyarrow.__version__; print('flash_attn', bool(importlib.util.find_spec('flash_attn')))"
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/qwen_env/" in line, f"Package leaked outside venv: {line}"
print("qwen_env ready:", PYTHON)
print("LOG:", LOG)


In [ ]:
# 2b. Extra pip (same slot as Video-LLaVA cell 5). Writes a log so a failure is inspectable.
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
LOG_DIR = ROOT / "outputs_reljoin_v6_qwen_multiseed" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / "cell5_pip.log"

if not PYTHON.exists():
    msg = (
        f"Missing {PYTHON}. Run the qwen_env bootstrap cell first "
        "(section 2), then re-run this cell.\n"
    )
    LOG.write_text(msg, encoding="utf-8")
    print(msg)
    raise FileNotFoundError(msg)

cmd = [
    str(PYTHON), "-m", "pip", "install",
    "--disable-pip-version-check",
    "--upgrade-strategy", "only-if-needed",
    "scikit-learn==1.3.2",
    "matplotlib==3.8.4",
    "numpy==1.25.0",
]
print("$", " ".join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_text = f"$ {' '.join(cmd)}\nexit={proc.returncode}\n\n{proc.stdout or ''}"
LOG.write_text(log_text, encoding="utf-8")
print(proc.stdout or "")
print("LOG:", LOG)

persist = Path(os.environ.get("RELJOIN_PERSIST", "/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs"))
if persist.parent.exists():
    try:
        persist.mkdir(parents=True, exist_ok=True)
        dest = persist / "logs" / LOG.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(log_text, encoding="utf-8")
        print("Copied log →", dest)
    except OSError as exc:
        print("Persist copy skipped:", exc)

if proc.returncode != 0:
    raise RuntimeError(f"cell 5 pip failed ({proc.returncode}). See {LOG}")
print("cell 5 pip OK")


## 3. Drive + pin HERBench evaluators


In [ ]:
# 3. Drive + pin HERBench evaluator (skip download if SHA matches)
from __future__ import annotations

import hashlib
import os
import time
import urllib.request
from pathlib import Path

ROOT = Path("/content")
VENV = ROOT / "qwen_env"
PYTHON = VENV / "bin" / "python"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "qwen25vl_7b"
OUT_DIR = ROOT / "outputs_reljoin_v6_qwen_multiseed"
EVAL_DIR = ROOT / "herbench_eval_pins"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
JSDELIVR_BASE = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/model_wrappers/base_vlm.py"
JSDELIVR_ACC = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/calculate_accuracy.py"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent, EVAL_DIR, OUT_DIR, OUT_DIR / "predictions"]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    RELJOIN_PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
if RELJOIN_PERSIST.resolve() == OUT_DIR.resolve():
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
    print("Persist dir was out_dir; using", RELJOIN_PERSIST)
RELJOIN_PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(RELJOIN_PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_pinned(urls: list[str], dest: Path, expected: str) -> None:
    if dest.exists() and sha256_file(dest) == expected:
        print("Already pinned:", dest.name)
        return
    last_err = None
    for url in urls:
        for attempt in range(4):
            try:
                print(f"GET {url} (try {attempt + 1})")
                urllib.request.urlretrieve(url, dest)
                got = sha256_file(dest)
                if got == expected:
                    print("OK", dest.name)
                    return
                last_err = RuntimeError(f"SHA mismatch for {dest.name}: {got}")
            except Exception as exc:
                last_err = exc
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Cannot pin {dest.name}: {last_err}")


fetch_pinned([HERBENCH_BASE_URL, JSDELIVR_BASE], EVAL_DIR / "base_vlm.py", HERBENCH_BASE_SHA256)
fetch_pinned([HERBENCH_ACC_URL, JSDELIVR_ACC], EVAL_DIR / "calculate_accuracy.py", HERBENCH_ACC_SHA256)

print({
    "RELJOIN_PERSIST": str(RELJOIN_PERSIST),
    "OUT_DIR": str(OUT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "EVAL_DIR": str(EVAL_DIR),
})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "qwen"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)


In [ ]:
# 4. Verify FrameWeave sources
from pathlib import Path
ROOT = Path("/content")
required = [
    ROOT / "run_reljoin.py",
    ROOT / "run_herbench_oracle_reljoin.py",
    ROOT / "run_nextgqa_reljoin.py",
    ROOT / "reljoin" / "__init__.py",
    ROOT / "reljoin" / "config.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "qwen_hooks.py",
    ROOT / "reljoin" / "qwen_generate.py",
    ROOT / "reljoin" / "unit_tests.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "data_perception.py",
    ROOT / "reljoin" / "frame_io.py",
    ROOT / "reljoin" / "losses.py",
    ROOT / "reljoin" / "diagnostics.py",
    ROOT / "reljoin" / "herbench_oracle.py",
    ROOT / "reljoin" / "stats.py",
    ROOT / "reljoin" / "nextgqa.py",
    ROOT / "reljoin" / "nextgqa_frames.py",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("All FrameWeave Hybrid sources present.")
for p in required:
    print(f"  OK {p} ({p.stat().st_size} bytes)")


## 5. Restore checkpoints


In [ ]:
# 5. Restore seed checkpoints and predictions from Drive
from __future__ import annotations
import shutil
from pathlib import Path

ROOT = Path("/content")
OUT_ROOT = ROOT / "outputs_reljoin_v6_qwen_multiseed"
try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]
try:
    CKPT_NAMES
except NameError:
    CKPT_NAMES = ["GlobalTimeFix.pt", "HybridNoGlobal.pt", "Hybrid.pt"]
try:
    PERSIST
except NameError:
    PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs")

for seed in SEEDS:
    src = PERSIST / f"seed_{seed}"
    dst = OUT_ROOT / f"seed_{seed}"
    if src.exists() and src.resolve() != dst.resolve():
        print(f"restore {src} -> {dst}")
        shutil.copytree(src, dst, dirs_exist_ok=True)
    ckpt_dir = dst / "checkpoints" / "best_external_val"
    missing = [name for name in CKPT_NAMES if not (ckpt_dir / name).is_file()]
    print({"seed": seed, "ckpt_dir": str(ckpt_dir), "missing": missing})
    assert not missing, (
        f"seed {seed} missing {missing} in {ckpt_dir}. "
        "Train notebook must finish this seed, and Drive must contain seed_{seed}/."
    )


## 6. Download Qwen2.5-VL-7B + HERBench lite


In [ ]:
# 6. Download Qwen2.5-VL-7B + HERBench lite (eval only)
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'qwen_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "qwen25vl_7b"
HERBENCH_HF_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_HF_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_REVISION = "cc594898137f460bfe9f0759e9844b3ce807cfb5"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent]:
    p.mkdir(parents=True, exist_ok=True)

print("Downloading Qwen2.5-VL-7B (~18 GB bf16)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

print("Downloading HERBench lite (~35GB)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{HERBENCH_HF_ID}',
    repo_type='dataset',
    revision='{HERBENCH_REVISION}',
    local_dir=r'{HERBENCH_ROOT}',
    allow_patterns=[
        'data/herbench_lite_v2.parquet',
        'data/herbench_annotations_lite_v2.json',
        'data/*metadata.json',
        'videos/videos.tar.part.00',
        'videos/videos.tar.part.01',
        'videos/videos.tar.part.02',
        'videos/videos.tar.part.03',
        'videos/videos_lite_info.txt',
        'videos/videos.tar.checksums.txt',
    ],
)
print('herbench snapshot OK')
"""], check=True)

LITE_MP4_MIN = 68
parts = sorted(HERBENCH_VIDEOS.glob("videos.tar.part.*"))
print("parts:", [p.name for p in parts])
mp4s = list(HERBENCH_VIDEOS.rglob("*.mp4"))
assert len(parts) >= 4 or len(mp4s) >= 2, (
    f"Thiếu videos.tar.part.00–03 trong {HERBENCH_VIDEOS}. Có: {[p.name for p in parts]}"
)
if len(mp4s) < LITE_MP4_MIN:
    assert len(parts) >= 4, (
        f"Chỉ có {len(mp4s)} mp4 và thiếu tar parts — tải lại videos.tar.part.00–03."
    )
    cat = subprocess.Popen(["cat", *map(str, parts)], stdout=subprocess.PIPE)
    subprocess.run(["tar", "-xf", "-", "-C", str(HERBENCH_VIDEOS)], stdin=cat.stdout, check=True)
    cat.wait()
    print("Extracted archives")
else:
    print(f"Already have {len(mp4s)} mp4")

n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4")))
if n_mp4 >= LITE_MP4_MIN:
    for part in parts:
        part.unlink(missing_ok=True)
    print("Removed archives after full extract")
else:
    print(f"KEEP tar parts — only {n_mp4} mp4 (need ~{LITE_MP4_MIN} for full eval)")

print({
    "videos_mp4": n_mp4,
    "model_config": (MODEL_DIR / "config.json").exists(),
    "free_GiB": round(shutil.disk_usage(ROOT).free / 2**30, 2),
})
assert (MODEL_DIR / "config.json").exists(), "Qwen config.json missing — tải lại cell 6."
assert n_mp4 >= 2, "Lite extract xong vẫn thiếu mp4"
if n_mp4 < LITE_MP4_MIN:
    print(f"WARN: {n_mp4} mp4 < {LITE_MP4_MIN}. Smoke có thể chạy; full eval sẽ thiếu video.")
print("Downloads OK")


## 7. Eval each seed


In [ ]:
# 7. HERBench lite_v2 for each seed. No training.
from __future__ import annotations
import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
RUNNER = ROOT / "run_herbench_oracle_reljoin.py"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "qwen25vl_7b"
EVAL_DIR = ROOT / "herbench_eval_pins"
OUT_ROOT = ROOT / "outputs_reljoin_v6_qwen_multiseed"
try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]
try:
    METHODS
except NameError:
    METHODS = "global_timefix,hybrid_noglobal,hybrid"
try:
    PERSIST
except NameError:
    PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs")

HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_REVISION = "cc594898137f460bfe9f0759e9844b3ce807cfb5"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"

assert PYTHON.exists() and RUNNER.is_file()
assert (MODEL_DIR / "config.json").exists(), "Run the download cell first."
n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4"))) if HERBENCH_VIDEOS.exists() else 0
assert n_mp4 >= 2, f"Need HERBench videos, found {n_mp4}"

for seed in SEEDS:
    seed_dir = OUT_ROOT / f"seed_{seed}"
    ckpt = seed_dir / "checkpoints" / "best_external_val"
    pred_dir = seed_dir / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    drive_pred = PERSIST / f"seed_{seed}" / "predictions"
    if PERSIST.exists():
        drive_pred.mkdir(parents=True, exist_ok=True)
        persist_dir = drive_pred
    else:
        persist_dir = pred_dir
    cmd = [
        str(PYTHON), "-u", str(RUNNER),
        "--root", "/content",
        "--herbench-root", str(HERBENCH_ROOT),
        "--videos-root", str(HERBENCH_VIDEOS),
        "--model-dir", str(MODEL_DIR),
        "--out-dir", str(pred_dir),
        "--persist-dir", str(persist_dir),
        "--herbench-revision", HERBENCH_REVISION,
        "--model-revision", MODEL_REVISION,
        "--herbench-base-vlm", str(EVAL_DIR / "base_vlm.py"),
        "--herbench-accuracy", str(EVAL_DIR / "calculate_accuracy.py"),
        "--herbench-base-sha256", HERBENCH_BASE_SHA256,
        "--herbench-acc-sha256", HERBENCH_ACC_SHA256,
        "--methods", METHODS,
        "--reljoin-ckpt", str(ckpt),
    ]
    log_path = OUT_ROOT / "logs" / f"eval_seed{seed}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["RELJOIN_ROOT"] = "/content"
    print("$", " ".join(cmd))
    with log_path.open("w", encoding="utf-8") as logf:
        proc = subprocess.Popen(
            cmd, cwd="/content", env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            logf.write(line)
            logf.flush()
        proc.wait()
    text = log_path.read_text(encoding="utf-8", errors="replace")
    if proc.returncode != 0 or "FULL OK" not in text:
        tail = "\n".join(text.splitlines()[-40:])
        raise RuntimeError(f"eval seed {seed} failed (exit {proc.returncode})\nLOG={log_path}\n\n{tail}")
    csvs = list(pred_dir.glob("predictions_*.csv"))
    assert csvs, f"no predictions csv in {pred_dir}"
    print(f"seed {seed} predictions:", [p.name for p in csvs])
    if persist_dir.resolve() != pred_dir.resolve():
        shutil.copytree(pred_dir, drive_pred, dirs_exist_ok=True)


## 8. Mean ± std


In [ ]:
# 8. Per-seed scores and mean ± sample std
from __future__ import annotations
import json
import math
from pathlib import Path
import pandas as pd

OUT_ROOT = Path("/content/outputs_reljoin_v6_qwen_multiseed")
SEEDS = [1, 100, 2027]
METHODS = ["global_timefix", "hybrid_noglobal", "hybrid"]
LABEL = {
    "global_timefix": "GlobalTimeFix (local off)",
    "hybrid_noglobal": "HybridNoGlobal (global off)",
    "hybrid": "Hybrid",
}
TSO = "Temporal Shot Ordering"
TASKS = [
    "Temporal Shot Ordering",
    "Multi-Person Duration Reasoning",
    "Action Sequence Integrity Identification",
    "Appearance-Grounded Behavior & Interactions",
    "Appearance-Grounded Attribute Recognition",
    "Appearance-Grounded Localization & Trajectory",
    "False Action Memory",
    "Scene Verification & Arrangement",
    "False Object Memory",
    "Multi-Entities Grounding and Localization",
    "Action Counting",
    "Region Localized People Counting",
]
FAMILIES = {
    "TR&C": TASKS[0:3],
    "R&T": TASKS[3:6],
    "GC&V": TASKS[6:9],
    "ME&N": TASKS[9:12],
}

def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        if isinstance(value, float) and math.isnan(value):
            return False
        return bool(value)
    return str(value).strip().lower() in {"1", "true", "yes", "1.0"}

def pct(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    return f"{100.0 * float(x):.2f}"

def summarize(df: pd.DataFrame) -> dict:
    df = df[df["status"].astype(str) == "ok"].copy() if "status" in df.columns else df.copy()
    df["is_correct"] = df["is_correct"].map(as_bool)
    per_task = {}
    for task in TASKS:
        sub = df[df["task_type"].astype(str) == task]
        per_task[task] = float(sub["is_correct"].mean()) if len(sub) else None
    missing = [t for t, v in per_task.items() if v is None]
    if missing:
        raise RuntimeError(f"missing tasks {missing}; saw {sorted(df['task_type'].astype(str).unique())}")
    non_tso = [per_task[t] for t in TASKS if t != TSO]
    families = {
        name: sum(per_task[t] for t in members) / len(members)
        for name, members in FAMILIES.items()
    }
    return {
        "n": int(len(df)),
        "QA": float(df["is_correct"].mean()) if len(df) else None,
        "Overall": sum(per_task.values()) / len(TASKS),
        "TSO": per_task[TSO],
        "Non-TSO": sum(non_tso) / len(non_tso),
        **families,
    }

rows = []
by_method = {m: [] for m in METHODS}
for seed in SEEDS:
    csvs = sorted((OUT_ROOT / f"seed_{seed}" / "predictions").glob("predictions_*.csv"))
    assert csvs, f"no predictions for seed {seed}"
    df = pd.read_csv(csvs[-1])
    for method in METHODS:
        sub = df[df["method"].astype(str) == method]
        assert len(sub), f"seed {seed} has no rows for {method}"
        stats = summarize(sub)
        stats["seed"] = seed
        stats["method"] = LABEL[method]
        rows.append(stats)
        by_method[method].append(stats)
        print(f"seed {seed} {method} n={stats['n']} QA={pct(stats['QA'])} Overall={pct(stats['Overall'])}")

metric_keys = ["Overall", "QA", "TSO", "Non-TSO", "TR&C", "R&T", "GC&V", "ME&N"]
for method in METHODS:
    group = by_method[method]
    mean = {"seed": "mean", "method": LABEL[method]}
    std = {"seed": "std", "method": LABEL[method]}
    complete = len(group) == len(SEEDS)
    for key in metric_keys:
        vals = [g[key] for g in group]
        mean[key] = sum(vals) / len(vals)
        if len(vals) >= 2:
            var = sum((v - mean[key]) ** 2 for v in vals) / (len(vals) - 1)
            std[key] = math.sqrt(var)
        else:
            std[key] = None
            complete = False
    mean["Complete"] = "yes" if complete else "no"
    std["Complete"] = mean["Complete"]
    rows.append(mean)
    rows.append(std)

table = pd.DataFrame(rows)
show = ["seed", "method", "Complete", "n"] + metric_keys
for key in metric_keys:
    table[key] = table[key].map(lambda v: pct(v) if not isinstance(v, str) else v)
keep = [c for c in show if c in table.columns]
out_dir = OUT_ROOT / "aggregate"
out_dir.mkdir(parents=True, exist_ok=True)
table.to_csv(out_dir / "results.csv", index=False)
view = table[keep].fillna("")
header = "| " + " | ".join(keep) + " |"
sep = "| " + " | ".join("---" for _ in keep) + " |"
body = ["| " + " | ".join(str(v) for v in row) + " |" for row in view.itertuples(index=False, name=None)]
(out_dir / "results.md").write_text("\n".join([header, sep, *body]) + "\n", encoding="utf-8")
print(view.to_string(index=False))
print("wrote", out_dir / "results.csv")
print("std is the sample standard deviation (ddof=1).")
print("Overall is the unweighted mean of 12 tasks. QA is micro accuracy. Non-TSO is the unweighted mean of 11.")


## 9. Export


In [ ]:
# 9. Copy eval artifacts to Drive and zip
from __future__ import annotations
import shutil
from pathlib import Path

OUT_ROOT = Path("/content/outputs_reljoin_v6_qwen_multiseed")
try:
    PERSIST
except NameError:
    PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs")
try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]

if PERSIST.exists():
    agg = OUT_ROOT / "aggregate"
    if agg.exists():
        shutil.copytree(agg, PERSIST / "aggregate", dirs_exist_ok=True)
        print("copied", agg)
    for seed in SEEDS:
        pred = OUT_ROOT / f"seed_{seed}" / "predictions"
        if pred.exists():
            shutil.copytree(pred, PERSIST / f"seed_{seed}" / "predictions", dirs_exist_ok=True)
            print("copied", pred)
zip_path = Path("/content/qwen_multiseed_eval")
shutil.make_archive(str(zip_path), "zip", OUT_ROOT / "aggregate")
print("zip", zip_path.with_suffix(".zip"))
try:
    from google.colab import files
    files.download(str(zip_path.with_suffix(".zip")))
except Exception as exc:
    print("download skipped:", exc)
